# TP1 – Continual Learning on Seq-CIFAR-10
**I309 – Visión Artificial Avanzada · Universidad de San Andrés**

This notebook is the single entry point for all experiments.
It follows the four stages of the assignment:

1. Dataset preparation (Seq-CIFAR-10, replay buffer)
2. Pre-training with Supervised Contrastive Learning (SupCon)
3. Continual Learning methods: Naive, EWC, LwF, Co²L
4. Comparison of results (Class-IL and Task-IL)


## 0. Setup

In [ ]:
import sys
import os

# Add repo root to path so all modules are importable
REPO_ROOT = os.path.dirname(os.path.abspath('__file__'))
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

import torch
print(f'PyTorch {torch.__version__}')
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')

In [ ]:
# Verify all module imports work
from data.dataset import SeqCIFAR10, ReplayBuffer, TASK_CLASSES, CLASS_NAMES
from models.backbone import Backbone
from losses.supcon import SupConLoss
from losses.distillation import AsymmetricDistillationLoss
from methods.base import BaseMethod
from methods.naive import NaiveFineTuning
from methods.ewc import EWC
from methods.lwf import LwF
from methods.co2l import Co2L
from utils.metrics import evaluate_class_il, evaluate_task_il, compute_forgetting
from utils.visualization import plot_accuracy_curve, plot_forgetting_curve, plot_embeddings, plot_loss_curve

print('All imports OK')

## Stage 1 – Dataset Preparation

Split CIFAR-10 into 5 sequential tasks (2 classes each) and set up the replay buffer.

| Task | Classes               | Train images |
|------|-----------------------|--------------|
| 0    | airplane, automobile  | 10 000       |
| 1    | bird, cat             | 10 000       |
| 2    | deer, dog             | 10 000       |
| 3    | frog, horse           | 10 000       |
| 4    | ship, truck           | 10 000       |

In [ ]:
DATA_ROOT = '../cifar-10'
N_TASKS   = 5
BATCH_SIZE = 128
BUFFER_SIZE = 200   # replay buffer capacity (total across all past tasks)

seq_cifar = SeqCIFAR10(data_root=DATA_ROOT, n_tasks=N_TASKS, batch_size=BATCH_SIZE)
buffer    = ReplayBuffer(capacity=BUFFER_SIZE)

print(f'Task splits: {TASK_CLASSES}')
print(f'Classes: {CLASS_NAMES}')

## Stage 2 – Pre-training with Supervised Contrastive Learning

Train the backbone encoder on Task 0 using SupCon loss, then attach and train a linear classification head.

In [ ]:
SUPCON_EPOCHS = 200
SUPCON_LR     = 0.5

backbone = Backbone(num_classes=10).to(DEVICE)
supcon_loss = SupConLoss(temperature=0.07)

# TODO: train encoder on Task 0 with SupCon
# TODO: plot loss curve and t-SNE embeddings at init / mid / final
# TODO: freeze encoder, train linear head, report Task-0 accuracy

## Stage 3 – Continual Learning Methods

Using the pre-trained backbone as the starting point, train each method
sequentially on tasks 0–4 and record Class-IL and Task-IL accuracy after each task.

In [ ]:
# --- 3.1 Naive Fine-Tuning ---
import copy
naive_backbone = copy.deepcopy(backbone)
naive = NaiveFineTuning(naive_backbone, device=DEVICE)

naive_class_il = []
naive_task_il  = []

for task_id in range(N_TASKS):
    train_loader = seq_cifar.get_task_loader(task_id, train=True)
    test_loaders = [seq_cifar.get_task_loader(t, train=False) for t in range(task_id + 1)]
    naive.begin_task(task_id)
    naive.train_task(task_id, train_loader, n_epochs=50)
    naive.end_task(task_id, train_loader)
    naive_class_il.append(evaluate_class_il(naive.backbone, test_loaders, DEVICE)['avg_acc'])
    naive_task_il.append(evaluate_task_il(naive.backbone, test_loaders, DEVICE)['avg_acc'])

In [ ]:
# --- 3.2 EWC ---
# TODO: repeat training loop with EWC method

In [ ]:
# --- 3.3 LwF ---
# TODO: repeat training loop with LwF method

In [ ]:
# --- 3.4 Co²L ---
# TODO: repeat training loop with Co2L method

## Stage 4 – Comparison of Results

In [ ]:
# Accuracy curves
all_class_il = {
    'Naive': naive_class_il,
    # 'EWC':   ewc_class_il,
    # 'LwF':   lwf_class_il,
    # 'Co2L':  co2l_class_il,
}
all_task_il = {
    'Naive': naive_task_il,
}

plot_accuracy_curve(all_class_il, scenario='Class-IL')
plot_accuracy_curve(all_task_il,  scenario='Task-IL')

In [ ]:
# Summary table
import pandas as pd

summary = pd.DataFrame({
    'Method':   list(all_class_il.keys()),
    'Class-IL': [v[-1] for v in all_class_il.values()],
    'Task-IL':  [v[-1] for v in all_task_il.values()],
})
summary